# Import thư viện

In [37]:
import paho.mqtt.client as mqtt
import json
import numpy as np
import pandas as pd
from collections import deque
from joblib import load
from datetime import datetime
import time
import os
import threading

In [ ]:
BROKER = "Enter your Broker ID"
PORT = 1883
TOPIC = "user+/data"
CLIENT_ID = "ml-predictor"

In [39]:
mqtt_node_red = mqtt.Client()
mqtt_node_red.connect(BROKER, 1883, 60)  # sửa localhost thành IP broker nếu cần

C:\Users\Admin\AppData\Local\Temp\ipykernel_8320\1320318190.py:1: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  mqtt_node_red = mqtt.Client()


<MQTTErrorCode.MQTT_ERR_SUCCESS: 0>

# Load mô hình

In [ ]:
# Số lượng user
user_ids = ["user1", "user2"]
num_user = len(user_ids)

# Đường dẫn gốc chứa mô hình và params
BASE_DIR = r"D:\Phenikaa\3rd Year - 1st Semester 2025\Machine Learning\End Term\26_10_2025_params"

# Biến lưu tất cả mô hình & tham số
models = {}
params_dict = {}

print("🔄 Đang load model và tham số cho tất cả user...")

try:
    # --- Load 1 model và 1 bộ params chung ---
    model_path = os.path.join(BASE_DIR, "logistic_reg_embbed.joblib")
    params_path = os.path.join(BASE_DIR, "z_params.json")

    model = load(model_path)
    with open(params_path, "r") as f:
        params = json.load(f)

    train_mean = np.array(params["mean"])
    train_std = np.array(params["std"])

    # --- Gán cho tất cả user ---
    for user_id in user_ids:
        models[user_id] = model
        params_dict[user_id] = {"mean": train_mean, "std": train_std}
        print(f"✅ {user_id}: Model và params loaded thành công!")

except Exception as e:
    print(f"❌ Lỗi load model hoặc params: {e}")

# Kiểm tra kết quả
print(f"\nTổng số mô hình đã load: {len(models)}")
print(f"models keys: {list(models.keys())}")
print(f"params_dict keys: {list(params_dict.keys())}")


🔄 Đang load model và tham số cho tất cả user...
✅ user1: Model và params loaded thành công!
✅ user2: Model và params loaded thành công!

📊 Tổng số mô hình đã load: 2
🔑 models keys: ['user1', 'user2']
🔑 params_dict keys: ['user1', 'user2']


# Trích xuất đặc trưng

In [41]:
def extract_features_from_array(data_array, feature_set):
    """Trích xuất features từ cửa sổ dữ liệu"""
    data = pd.DataFrame(data_array, columns=['ax', 'ay', 'az', 'gx', 'gy', 'gz'])
    features = []
    available_features = {
        'mean': np.mean,
        'std': np.std,
        'min': np.min,
        'max': np.max,
        'skew': lambda x: np.mean((x - np.mean(x))**3) / (np.std(x)**3 + 1e-10),
        'kurtosis': lambda x: np.mean((x - np.mean(x))**4) / (np.std(x)**4 + 1e-10),
    }
    for feature_name in feature_set:
        if feature_name in available_features:
            feature_func = available_features[feature_name]
            features.extend(data.apply(feature_func, axis=0).values)
    return np.array(features)

In [42]:
# --- Tham số toàn cục (dùng chung) ---
WINDOW = 40
OVERLAP = 0.85
STEP = int(WINDOW * (1 - OVERLAP))
SAMPLING_RATE = 20  # Hz

feature_set = ['mean', 'std', 'min', 'max', 'skew', 'kurtosis']
label_map = {
    0: 'walking',
    1: 'standing/sitting',
    2: 'lying',
    3: 'standup/sitdown',
    4: 'fall',
    5: 'unknown'
}

# --- Trạng thái riêng cho từng user ---
user_data = {
    "user1": {
        "buffer": [],
        "sample_count": 0,
        "prediction_count": 0,
        "window_start_time": time.time(),
        "last_prediction": None,
        "last_confidence": 0
    },
    "user2": {
        "buffer": [],
        "sample_count": 0,
        "prediction_count": 0,
        "window_start_time": time.time(),
        "last_prediction": None,
        "last_confidence": 0
    },
    
}

# --- Tổng thống kê hệ thống (không bắt buộc) ---
prediction_count = 0


In [43]:
def make_prediction(user_id):
    """
    Thực hiện prediction cho từng user khi đủ WINDOW samples.
    """
    global prediction_count, user_data, window_start_time, params_dict, models


    try:
        # --- Đảm bảo user_id tồn tại trong user_data ---
        if user_id not in user_data:
            print(f"⚠️ user_id '{user_id}' chưa được khởi tạo trong user_data. Khởi tạo mới...")
            user_data[user_id] = {
                "buffer": [],
                "sample_count": 0,
                "prediction_count": 0,
                "window_start_time": time.time(),
                "last_prediction": None,
                "last_confidence": 0
            }
            return None, None  # chưa có dữ liệu thì không predict

        # --- Lấy dữ liệu riêng của user ---
        user_state = user_data[user_id]
        data_buffer = user_state["buffer"]
        last_prediction = user_state["last_prediction"]
        last_confidence = user_state["last_confidence"]

        # --- Kiểm tra buffer ---
        if len(data_buffer) < WINDOW:
            print(f"⚠️  User {user_id}: Buffer chưa đủ ({len(data_buffer)}/{WINDOW})")
            return None, None

        # --- Tạo feature từ buffer ---
        window_data = data_buffer[:WINDOW]
        data_array = np.array(window_data)
        features = extract_features_from_array(data_array, feature_set).reshape(1, -1)

        # --- Normalize ---
        mean = params_dict[user_id]["mean"]
        std = params_dict[user_id]["std"]
        normalized = np.where(std == 0, 0, (features - mean) / std)

        # --- Predict ---
        model = models[user_id]
        probs = model.predict_proba(normalized)[0]
        confidence = np.max(probs)

        pred_label = np.argmax(probs) if confidence >= 0.5 else 5

        # --- Hiển thị kết quả ---
        user_state["prediction_count"] += 1
        prediction_count += 1
        is_changed = (pred_label != last_prediction)

        print(f"\n📊 [User {user_id}] Prediction #{user_state['prediction_count']}")
        if is_changed and last_prediction is not None:
            print(f"Activity: {label_map[pred_label].upper()} ({confidence*100:.1f}%) ⬅️ From {label_map[last_prediction]}")
            

        else:
            print(f"Activity: {label_map[pred_label].upper()} ({confidence*100:.1f}%)")
        print("="*60)

        # --- Top 3 confidence scores ---
        top3_idx = np.argsort(probs)[-3:][::-1]
        for idx in top3_idx:
            print(f"   {label_map[idx]:10s}: {probs[idx]*100:5.1f}%")

        # --- Gửi về Node-RED ---
        payload = {
                "prediction": label_map[pred_label],
                "confidence": float(confidence),
                "timestamp": int(time.time())
            }
        mqtt_node_red.publish(f"{user_id}/output", json.dumps(payload))  # ✅ Không thêm 'user' hai lần   
        
        # --- Cập nhật state ---
        user_state["buffer"] = user_state["buffer"][STEP:]
        user_state["last_prediction"] = pred_label
        user_state["last_confidence"] = confidence
        user_state["window_start_time"] = time.time()

        return pred_label, confidence

    except KeyError as e:
        print(f"❌ KeyError trong make_prediction(): {e}")
        print(f"🔍 user_data keys hiện có: {list(user_data.keys())}")
        return None, None

    except Exception as e:
        print(f"❌ Lỗi prediction cho user {user_id}: {e}")
        return None, None


In [44]:
def create_user_client(user_id):
    """Khởi tạo MQTT client riêng cho mỗi user"""
    
    def on_connect(client, userdata, flags, reason_code, properties=None):
        if reason_code == 0:
            topic = f"{user_id}/data"
            client.subscribe(topic)
            print(f"✅ [{user_id}] Connected & subscribed to {topic}")
        else:
            print(f"❌ [{user_id}] Connect failed: {reason_code}")

    def on_message(client, userdata, msg):
        try:
            data = json.loads(msg.payload.decode('utf-8'))
            sample = [data.get(k, 0) for k in ['ax', 'ay', 'az', 'gx', 'gy', 'gz']]

            user_buffer = user_data[user_id]["buffer"]
            user_buffer.append(sample)
            
            if len(user_buffer) >= WINDOW:
                make_prediction(user_id)
                user_buffer = user_buffer[STEP:]  # giữ overlap

        except Exception as e:
            print(f"❌ [{user_id}] Error processing message: {e}")

    # Khởi tạo client riêng cho user
    client = mqtt.Client(userdata={"id": user_id})
    client.on_connect = on_connect
    client.on_message = on_message
    client.connect(BROKER, PORT)

    # Chạy thread riêng
    thread = threading.Thread(target=client.loop_forever, daemon=True)
    thread.start()

    return client, thread


In [ ]:
if __name__ == "__main__":
    print("\n" + "="*50)
    print("MULTI-USER MQTT + ML PIPELINE STARTED")
    print("="*50)
    print(f"Broker: {BROKER}:{PORT}")
    print(f"Window: {WINDOW} samples | Step: {STEP}")
    print(f"Users: {', '.join(user_ids)}")
    print("="*50 + "\n")

    clients = []
    try:
        # Khởi tạo client riêng cho mỗi user
        for uid in user_ids:
            c, _ = create_user_client(uid)
            clients.append(c)
        print("All clients are running. Waiting for data...\n")

        # Giữ chương trình chạy nhẹ
        while True:
            time.sleep(1)

    except KeyboardInterrupt:
        print("\nStopped by user.")
        for c in clients:
            c.disconnect()
        print("Disconnected all clients.")
    except Exception as e:
        print(f"❌ Main error: {e}")

C:\Users\Admin\AppData\Local\Temp\ipykernel_8320\1570915183.py:28: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client(userdata={"id": user_id})



🚀 MULTI-USER MQTT + ML PIPELINE STARTED
📍 Broker: 172.20.10.4:1883
🪟 Window: 40 samples | Step: 6
👥 Users: user1, user2

✅ [user1] Connected & subscribed to user1/data
✅ All clients are running. Waiting for data...

✅ [user2] Connected & subscribed to user2/data

📊 [User user1] Prediction #1
Activity: WALKING (63.5%)
   walking   :  63.5%
   standing/sitting:  22.5%
   standup/sitdown:  12.9%

📊 [User user1] Prediction #2
Activity: WALKING (59.6%)
   walking   :  59.6%
   standing/sitting:  25.3%
   standup/sitdown:  13.9%

📊 [User user1] Prediction #3
Activity: WALKING (64.5%)
   walking   :  64.5%
   standing/sitting:  21.9%
   standup/sitdown:  12.1%

📊 [User user1] Prediction #4
Activity: WALKING (54.5%)
   walking   :  54.5%
   standing/sitting:  27.9%
   standup/sitdown:  15.4%

📊 [User user1] Prediction #5
Activity: STANDING/SITTING (50.3%) ⬅️ From walking
   standing/sitting:  50.3%
   standup/sitdown:  25.8%
   walking   :  21.2%

📊 [User user1] Prediction #6
Activity: STANDI